# BDA POC — Long, mixed-content PDF (text + tables + images)
### SageMaker · Standard output with element extraction + splitter

Companion to the first BDA POC. This one is tuned for a **single long PDF that mixes body text, embedded figures/images, and tables** — e.g. a research report, prospectus, annual report, or a multi-doc application packet.

**What you get out the other end:**
- Full document **text** (per-page markdown + a generative document summary)
- Every **table** as a `pandas.DataFrame` (from BDA's CSV/HTML table representation)
- Every **figure/embedded image** downloaded as an actual `.png` crop, with its generated caption
- Page-level **provenance** (page index + bounding box) on each element — ready for RAG citation

Two non-negotiable settings drive all of this: the **document splitter** (for length) and **`ELEMENT` granularity** (for tables/figures). Both are configured below.


## 0 · Limits that shape the design (current, async API)

| Constraint | Value | Consequence |
|---|---|---|
| Max pages / document (no splitter) | **20** | A long PDF *must* use the splitter |
| Max pages / document (**splitter ENABLED**) | **3,000** | This is your real ceiling |
| Max file size (API) | **500 MB** (200 MB via console) | |
| Formats | PDF, TIFF, JPEG, PNG, DOCX | DOCX is converted to PDF (loses page mapping) |
| Figure captioning | 20 images / page (async) | |
| Languages | EN, DE, ES, FR, IT, PT | No vertical CJK text |
| **Custom blueprint** page cap | **~20 pages / sub-document**, 100 fields max | Custom extraction is *per split sub-doc*, not whole-file |

**Takeaway:** standard output scales to 3,000 pages with the splitter. Custom-field extraction does not scale the same way — it runs per split sub-document. For a long continuous report, lean on **standard output element extraction** (below). For a long *packet* of distinct docs, use the splitter to route each sub-doc to a blueprint (shown in the optional section).


In [ ]:
%pip install --no-warn-conflicts "boto3>=1.37.6" sagemaker pandas openpyxl -Uqq

In [ ]:
import boto3, json, os, io, time
from urllib.parse import urlparse
from pathlib import Path
import pandas as pd

REGION = os.environ.get("AWS_REGION") or "ap-south-1"   # BDA is in Mumbai; override if needed
session     = boto3.Session(region_name=REGION)
sts         = session.client("sts")
s3_client   = session.client("s3")
bda_client  = session.client("bedrock-data-automation")
bda_runtime = session.client("bedrock-data-automation-runtime")
account_id  = sts.get_caller_identity()["Account"]

try:
    import sagemaker
    BUCKET = sagemaker.Session(boto_session=session).default_bucket()
except Exception:
    BUCKET = os.environ.get("BDA_BUCKET", "REPLACE-WITH-YOUR-BUCKET")

S3_INPUT, S3_OUTPUT = f"s3://{BUCKET}/bda/input", f"s3://{BUCKET}/bda/output"

def cris_prefix(r):
    if r.startswith("us-") or r.startswith("us_gov"): return "us"
    if r.startswith("eu-"): return "eu"
    if r.startswith("ap-"): return "apac"
    raise ValueError(f"Unknown geography for {r}")
PROFILE_ARN = f"arn:aws:bedrock:{REGION}:{account_id}:data-automation-profile/{cris_prefix(REGION)}.data-automation-v1"

print("Region:", REGION, "| Bucket:", BUCKET, "| Profile:", PROFILE_ARN)

## 1 · Helpers (self-contained)

In [ ]:
def split_s3(uri):
    p = urlparse(uri); return p.netloc, p.path.lstrip("/")

def read_s3_json(uri):
    b, k = split_s3(uri); return json.loads(s3_client.get_object(Bucket=b, Key=k)["Body"].read())

def read_s3_bytes(uri):
    b, k = split_s3(uri); return s3_client.get_object(Bucket=b, Key=k)["Body"].read()

def upload(local, s3_uri):
    b, k = split_s3(s3_uri); s3_client.upload_file(local, b, k); return s3_uri

def wait_for_invocation(arn, delay=20, max_iter=60):
    for _ in range(max_iter):
        r = bda_runtime.get_data_automation_status(invocationArn=arn); st = r["status"]
        if st == "Success": print("  ->", st); return r
        if st in ("ClientError", "ServiceError"):
            raise RuntimeError(f"{st}: {r.get('error_type')} / {r.get('error_message')}")
        print("  ...", st); time.sleep(delay)
    raise TimeoutError("job did not complete")

def invoke(input_s3, project_arn, stage="LIVE"):
    arn = bda_runtime.invoke_data_automation_async(
        inputConfiguration={"s3Uri": input_s3},
        outputConfiguration={"s3Uri": S3_OUTPUT},
        dataAutomationProfileArn=PROFILE_ARN,
        dataAutomationConfiguration={"dataAutomationProjectArn": project_arn, "stage": stage},
    )["invocationArn"]
    print("  invocation:", arn.split("/")[-1]); return wait_for_invocation(arn)

def all_segments(status_response):
    """Splitter produces MANY segments. Return list of (standard_json, custom_json_or_None)."""
    meta = read_s3_json(status_response["outputConfiguration"]["s3Uri"])
    segs = []
    for asset in meta.get("output_metadata", []):
        for seg in asset.get("segment_metadata", []):
            std  = read_s3_json(seg["standard_output_path"]) if seg.get("standard_output_path") else None
            cust = (read_s3_json(seg["custom_output_path"])
                    if seg.get("custom_output_status") == "MATCH" and seg.get("custom_output_path") else None)
            segs.append((std, cust))
    print(f"  {len(segs)} segment(s) returned by the splitter")
    return segs

## 2 · Standard output config — every toggle explained

Each switch below maps directly to something you'll want from a mixed-content doc:

| Setting | Gives you |
|---|---|
| `granularity: ELEMENT` | **tables & figures as discrete elements** (the whole point) — plus DOCUMENT/PAGE/LINE/WORD levels |
| `outputFormat.textFormat: [MARKDOWN, HTML, CSV, PLAIN_TEXT]` | tables arrive with `representation.csv` / `.html` → straight into pandas |
| `additionalFileFormat: ENABLED` | BDA writes **figure crop images** and **per-table CSV files** into your output bucket |
| `generativeField: ENABLED` | document summary, **table summaries**, and **figure captions** |
| `boundingBox: ENABLED` | page index + box per element → RAG provenance / overlays |
| `overrideConfiguration.splitter: ENABLED` | unlocks the **3,000-page** ceiling |


In [ ]:
standard_output_config = {
    "document": {
        "extraction": {
            "granularity": {"types": ["DOCUMENT", "PAGE", "ELEMENT", "LINE", "WORD"]},
            "boundingBox": {"state": "ENABLED"}
        },
        "generativeField": {"state": "ENABLED"},
        "outputFormat": {
            "textFormat": {"types": ["PLAIN_TEXT", "MARKDOWN", "HTML", "CSV"]},
            "additionalFileFormat": {"state": "ENABLED"}
        }
    }
}
# THE setting that makes long PDFs work:
override_configuration = {"document": {"splitter": {"state": "ENABLED"}}}

In [ ]:
PROJECT_NAME = "poc-long-pdf-standard"
existing = next((p for p in bda_client.list_data_automation_projects(projectStageFilter="LIVE").get("projects", [])
                 if p["projectName"] == PROJECT_NAME), None)
if existing:
    proj = bda_client.update_data_automation_project(
        projectArn=existing["projectArn"],
        standardOutputConfiguration=standard_output_config,
        overrideConfiguration=override_configuration)
else:
    proj = bda_client.create_data_automation_project(
        projectName=PROJECT_NAME, projectStage="LIVE",
        projectDescription="Long mixed-content PDF: text + tables + figures, splitter on",
        standardOutputConfiguration=standard_output_config,
        overrideConfiguration=override_configuration)
project_arn = proj["projectArn"]
print("Project ARN:", project_arn)

## 3 · Point at your long PDF and invoke

In [ ]:
DOC_LOCAL = "data/long_document.pdf"     # <- drop your long PDF here
DOC_S3    = None                          # <- or set an existing s3:// URI

os.makedirs("data", exist_ok=True)
if DOC_S3 is None:
    assert Path(DOC_LOCAL).exists(), f"Put a PDF at {DOC_LOCAL} (or set DOC_S3)"
    DOC_S3 = upload(DOC_LOCAL, f"{S3_INPUT}/{Path(DOC_LOCAL).name}")
print("Input:", DOC_S3)

status = invoke(DOC_S3, project_arn)
segments = all_segments(status)
standard_outputs = [std for std, _ in segments if std]

## 4 · Text — assemble the full document

With the splitter on, the doc may come back as several segments. Concatenate page markdown across all of them, and grab the generative summaries.

In [ ]:
full_markdown, doc_summaries = [], []
for std in standard_outputs:
    doc_summaries.append((std.get("document", {}) or {}).get("summary"))
    for page in std.get("pages", []):
        md_txt = (page.get("representation", {}) or {}).get("markdown", "")
        full_markdown.append(md_txt)

full_text = "\n\n".join(m for m in full_markdown if m)
print(f"Assembled {len(full_markdown)} page(s), {len(full_text):,} chars of markdown\n")
print("=== DOCUMENT SUMMARY (segment 0) ===")
print(doc_summaries[0] if doc_summaries else "(none)")
print("\n=== First 600 chars of body ===\n", full_text[:600])

## 5 · Tables → pandas DataFrames

Every `TABLE` element carries `representation.csv` (and `.html`). Parse to DataFrames; fall back to HTML for ragged/merged-cell tables. Optionally dump them all to one Excel workbook.

In [ ]:
def collect_tables(standard_outputs):
    tables = []
    for seg_i, std in enumerate(standard_outputs):
        for el in std.get("elements", []):
            if el.get("type") != "TABLE":
                continue
            rep = el.get("representation", {}) or {}
            df = None
            if rep.get("csv"):
                try: df = pd.read_csv(io.StringIO(rep["csv"]))
                except Exception: df = None
            if df is None and rep.get("html"):
                try: df = pd.read_html(io.StringIO(rep["html"]))[0]
                except Exception: df = None
            if df is None and el.get("csv_s3_uri"):
                try: df = pd.read_csv(io.BytesIO(read_s3_bytes(el["csv_s3_uri"])))
                except Exception: df = None
            loc = (el.get("locations") or [{}])[0]
            tables.append({
                "segment": seg_i,
                "page": loc.get("page_index", el.get("page_indices", [None])[0]),
                "title": el.get("title"),
                "summary": el.get("summary"),
                "df": df,
            })
    return tables

tables = collect_tables(standard_outputs)
print(f"Found {len(tables)} table(s)\n")
for t in tables[:3]:
    print(f"--- Table p{t['page']} | {t['title'] or '(untitled)'} ---")
    print(t["df"].head() if t["df"] is not None else "(could not parse)")
    print()

# Optional: one Excel workbook, one sheet per table
if tables:
    with pd.ExcelWriter("extracted_tables.xlsx", engine="openpyxl") as xl:
        for i, t in enumerate(tables):
            if t["df"] is not None:
                t["df"].to_excel(xl, sheet_name=f"p{t['page']}_t{i}"[:31], index=False)
    print("Wrote extracted_tables.xlsx")

## 6 · Figures / embedded images → downloaded crops

`FIGURE` elements carry `crop_images` (S3 URIs of the cropped image) and a generative `summary` (caption). Download each crop locally.

In [ ]:
os.makedirs("figures", exist_ok=True)
figures = []
for seg_i, std in enumerate(standard_outputs):
    for el in std.get("elements", []):
        if el.get("type") != "FIGURE":
            continue
        loc = (el.get("locations") or [{}])[0]
        page = loc.get("page_index", el.get("page_indices", [None])[0])
        for j, crop_uri in enumerate(el.get("crop_images", []) or []):
            local = f"figures/seg{seg_i}_p{page}_fig{j}.png"
            try:
                with open(local, "wb") as f: f.write(read_s3_bytes(crop_uri))
                figures.append({"page": page, "path": local, "caption": el.get("summary")})
            except Exception as e:
                print("  skip", crop_uri, e)

print(f"Downloaded {len(figures)} figure crop(s) to ./figures/\n")
for fig in figures[:5]:
    print(f"  p{fig['page']}: {fig['path']}  |  {fig['caption'] or '(no caption)'}")

# Preview one inline
if figures:
    from IPython.display import Image, display
    display(Image(filename=figures[0]["path"], width=350))

## 7 · (Optional) Custom blueprints on a long *packet* via splitter routing

If your long PDF is really a **packet of distinct documents** (e.g. an application bundle: transcript + score report + financial statement), attach up to **40 blueprints** to the same splitter-enabled project. BDA splits the file and routes each sub-document to the best-matching blueprint. Each sub-doc still obeys the ~20-page / 100-field custom cap — but the *packet* can be long.

`custom_output_status` per segment tells you `MATCH` (a blueprint fit) vs `NO_MATCH`.

In [ ]:
# Example: attach one custom blueprint to the same project so long packets get routed.
# (Reuse a blueprint ARN from the first POC notebook, or create one here.)
CUSTOM_BLUEPRINT_ARN = None   # <- set to an existing blueprint ARN to enable this section

if CUSTOM_BLUEPRINT_ARN:
    bda_client.update_data_automation_project(
        projectArn=project_arn,
        standardOutputConfiguration=standard_output_config,
        customOutputConfiguration={"blueprints": [
            {"blueprintArn": CUSTOM_BLUEPRINT_ARN, "blueprintStage": "LIVE"}]},
        overrideConfiguration=override_configuration)   # splitter stays ON
    status2 = invoke(DOC_S3, project_arn)
    for seg_i, (std, cust) in enumerate(all_segments(status2)):
        print(f"segment {seg_i}: custom={'MATCH' if cust else 'no match'}")
        if cust:
            print(json.dumps(cust.get("inference_result", cust), indent=2)[:600])
else:
    print("Set CUSTOM_BLUEPRINT_ARN to try packet routing.")

## 8 · RAG-ready, element-aware chunking

The reason you extracted elements instead of a blob: chunk **by element**, keep tables whole, and attach figure captions + page provenance. This preserves structure that naive character-splitting destroys — tables stay queryable, figures stay findable, and every chunk carries a page cite.

In [ ]:
def build_chunks(standard_outputs):
    chunks = []
    for seg_i, std in enumerate(standard_outputs):
        for el in std.get("elements", []):
            loc = (el.get("locations") or [{}])[0]
            page = loc.get("page_index", el.get("page_indices", [None])[0])
            rep = el.get("representation", {}) or {}
            etype = el.get("type")
            if etype == "TABLE":
                content = rep.get("markdown") or rep.get("html") or rep.get("csv") or ""
                content = f"[TABLE] {el.get('title') or ''}\n{content}\nSummary: {el.get('summary') or ''}"
            elif etype == "FIGURE":
                content = f"[FIGURE] {el.get('summary') or '(image)'}"
            else:
                content = rep.get("markdown") or rep.get("text") or ""
            if content.strip():
                chunks.append({"type": etype, "page": page, "segment": seg_i,
                               "reading_order": el.get("reading_order"), "content": content})
    return chunks

chunks = build_chunks(standard_outputs)
print(f"{len(chunks)} element-aware chunks "
      f"({sum(c['type']=='TABLE' for c in chunks)} tables, "
      f"{sum(c['type']=='FIGURE' for c in chunks)} figures, "
      f"{sum(c['type']=='TEXT' for c in chunks)} text)\n")
print(json.dumps(chunks[0], indent=2)[:500] if chunks else "no chunks")
# -> embed `content`, store {type, page, segment} as metadata in your vector store.

## 9 · Cleanup (optional)

In [ ]:
# bda_client.delete_data_automation_project(projectArn=project_arn)
# print("deleted project")

---
### Notes for your pipeline
- **Splitter is the whole ballgame for length.** Forget it and you're capped at 20 pages. It also means *always* loop over segments — never assume one result.
- **Standard output does the structural heavy lifting** (text + tables + figures). Reach for custom blueprints only when you need *specific named fields*, and remember they run per ≤20-page sub-doc.
- **`additionalFileFormat: ENABLED` costs S3 writes** (crop images + table CSVs land in your output bucket). If you only need inline `representation.csv`, you can leave it off and save the storage.
- **Generative fields cost latency.** If you don't need summaries/captions, disabling `generativeField` measurably speeds up big jobs.
- **Element-aware chunking >> character chunking** for RAG over reports — tables and figures survive as first-class, citable units.
- **ScoreNLearn fit:** an admissions "packet PDF" (transcript + scores + SOP + financials) is the packet-routing case (section 7); a single long university brochure or policy doc is the standard-element case (sections 4–6).
